# Silage — Flieg score (pH + dry matter)

`Flieg = 220 + (2 × DM% − 15) − 40 × pH`

Data fetched from processed Indian tables: Kaur 2026 Punjab n=100 summary,
Hundal 2020 wheat silage, Brar 2021 maize silage, `flieg_score_rules.csv`.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("../..").resolve()
if not (ROOT / "smartfeed").exists():
    ROOT = Path(".").resolve()
    while ROOT != ROOT.parent and not (ROOT / "smartfeed").exists():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data" / "processed"
print("ROOT", ROOT)
print("DATA", DATA)


ROOT /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111
DATA /Users/sayangarai/Documents/Programming/Projects/SIH_PS_260111/data/processed


In [2]:
from smartfeed.silage import assess_silage, flieg_score

kaur = pd.read_csv(DATA / "kaur_2026_punjab_farmer_silage_summary_n100.csv")
hundal = pd.read_csv(DATA / "hundal_2020_wheat_silage.csv")
brar = pd.read_csv(DATA / "brar_2021_punjab_maize_cultivar_silage.csv")
rules = pd.read_csv(DATA / "flieg_score_rules.csv")
print("Kaur summary")
print(kaur)
print("\nHundal rows", len(hundal), "Brar rows", len(brar))
print(rules)


Kaur summary
     parameter   unit    n   mean     sd    min     q1  median      q3     max
0           DM      %  100  29.52   1.30  26.25  28.65   29.48   30.35   32.65
1           pH     pH  100   4.34   0.93   3.10   3.61    3.99    4.90    7.47
2           CP      %  100   9.36   0.88   6.57   8.69    9.36    9.88   11.37
3  Lactic acid      %  100   4.34   0.58   2.83   4.04    4.30    4.61    6.52
4         NH3N      %  100   9.82   2.30   5.12   7.96    9.67   11.79   14.70
5  Fleig Point  score  100  90.58  36.82 -34.10  68.40  104.10  116.80  142.80

Hundal rows 8 Brar rows 3
   min_score  max_score         class  \
0         81       1000     very_good   
1         61         80          good   
2         41         60  satisfactory   
3         21         40      moderate   
4      -1000         20          poor   

                                              advice  
0  Good fermentation. Feed normally and keep silo...  
1  Acceptable quality. Use soon and avoid heating.

In [3]:
print("demo good", assess_silage(ph=3.9, moisture_pct=70, ingredient="maize silage"))
print("demo poor", assess_silage(ph=5.5, moisture_pct=78, ingredient="wheat silage"))
print("check formula vs Hundal row 0", hundal.iloc[0][["pH", "flieg_score"]].to_dict())


demo good {'present': True, 'pH': 3.9, 'dm_pct': 30.0, 'formula': '220 + (2 * DM% - 15) - 40 * pH', 'kaur_punjab_farmer_n100': 'median pH 3.99, Flieg 104 (summary, not raw pits)', 'flieg_score': 109.0, 'class': 'very_good', 'advice': 'Good fermentation. Feed normally and keep silo face clean.', 'flags': [], 'ingredient': 'maize silage'}
demo poor {'present': True, 'pH': 5.5, 'dm_pct': 22.0, 'formula': '220 + (2 * DM% - 15) - 40 * pH', 'kaur_punjab_farmer_n100': 'median pH 3.99, Flieg 104 (summary, not raw pits)', 'flieg_score': 29.0, 'class': 'moderate', 'advice': 'Poor fermentation. Limit feeding and check for mould.', 'flags': ['high_pH_low_DM'], 'ingredient': 'wheat silage'}
check formula vs Hundal row 0 {'pH': 4.15, 'flieg_score': 100.7}


In [4]:
art = Path("artifacts")
art.mkdir(exist_ok=True)
(art / "metrics.json").write_text(
    json.dumps(
        {
            "model": "flieg_from_pH_and_DM",
            "formula": "220 + (2*DM - 15) - 40*pH",
            "n_hundal": int(len(hundal)),
            "n_brar": int(len(brar)),
            "kaur_n": 100,
            "kaur_note": "percentiles, not raw pits",
        },
        indent=2,
    )
)
print("saved")


saved
